# Phase 3: EfficientAD comparison and robustness

Trains PatchCore and EfficientAD on the same splits, scores clean and perturbed
(brightness, blur, noise) test images with each model's clean threshold
and packs `results/` into one zip.

Runs on Google Colab (GPU runtime) or Kaggle (GPU + Internet on). No tokens or secrets
are used: the repository is cloned over public HTTPS. Every step stops the notebook on
the first error.

**Before running:** upload the MVTec archives as described in step 3 (see also `DATA.md`).

## 1. Environment

In [ ]:
import os
import subprocess
import sys

# Kaggle sets this variable; Colab does not (Colab may still have a /kaggle folder).
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
WORK_DIR = "/kaggle/working" if IN_KAGGLE else "/content"
print("Python", sys.version.split()[0], "| Kaggle" if IN_KAGGLE else "| Colab")


def run(*args: str) -> None:
    """Run a command, show its output, and stop the notebook if it fails."""
    print("$", " ".join(args), flush=True)
    result = subprocess.run(args)
    if result.returncode != 0:
        raise RuntimeError(f"Step failed (exit code {result.returncode}): {' '.join(args)}")


run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")

## 2. Clone and install (pinned versions from `pyproject.toml`)

If an earlier install in this runtime failed, start fresh first
(Runtime > Disconnect and delete runtime): a broken torch/torchvision pair stays installed.

In [ ]:
os.chdir(WORK_DIR)
run("rm", "-rf", "visual-quality-inspection")
run("git", "clone", "--depth", "1", "https://github.com/BurakAytas/visual-quality-inspection.git")
os.chdir("visual-quality-inspection")
run("git", "log", "-1", "--format=commit %H")
run(sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]")

# torch and torchvision must come from the same CUDA build, and a GPU must be visible.
run(
    sys.executable,
    "-c",
    "import torch, torchvision; "
    "print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))",
)

## 3. Data

Download all three archives yourself through the web forms on the MVTec website
(non-commercial licence), then upload them:

| Archive | Colab (Google Drive) | Kaggle (private dataset) |
|---|---|---|
| MVTec AD 2 `fabric` | `MyDrive/datasets/mvtec_ad_2/fabric.tar.gz` | `/kaggle/input/mvtec-ad-2/` |
| MVTec AD `leather`, `carpet` | `MyDrive/datasets/mvtec_ad/` (e.g. `leather.tar.xz`) | `/kaggle/input/mvtec-ad/` |

A source can be an archive, an unpacked folder, or a folder holding `<category>.tar*`.
Edit the two paths below if yours differ.

In [ ]:
if IN_KAGGLE:
    AD2_SOURCE = "/kaggle/input/mvtec-ad-2"
    AD_SOURCE = "/kaggle/input/mvtec-ad"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    AD2_SOURCE = "/content/drive/MyDrive/datasets/mvtec_ad_2/fabric.tar.gz"
    AD_SOURCE = "/content/drive/MyDrive/datasets/mvtec_ad"

run(
    sys.executable,
    "scripts/download_data.py",
    "--config",
    "configs/fabric.yaml",
    "--source",
    AD2_SOURCE,
)
run(
    sys.executable,
    "scripts/download_data.py",
    "--config",
    "configs/leather.yaml",
    "--source",
    AD_SOURCE,
)
run(
    sys.executable,
    "scripts/download_data.py",
    "--config",
    "configs/carpet.yaml",
    "--source",
    AD_SOURCE,
)

## 4. Train and score

For each model and category: fit on `train/good`, score the good-only validation split,
set the threshold at the 1% false-alarm quantile of those scores, then score the clean and
the perturbed test split and save overlays. Nothing here looks at test labels before the
threshold is fixed. EfficientAD downloads Imagenette and its teacher weights on first use.

In [ ]:
MODELS = ["patchcore", "efficient_ad"]
CATEGORIES = ["configs/fabric.yaml", "configs/leather.yaml", "configs/carpet.yaml"]
for model in MODELS:
    for config in CATEGORIES:
        run(
            sys.executable,
            "scripts/train_anomaly.py",
            "--config",
            config,
            "--model",
            f"configs/models/{model}.yaml",
        )

## 5. Metrics (CPU; reproducible later from `results/` alone)

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

for model in MODELS:
    run(
        sys.executable,
        "scripts/evaluate.py",
        "--model",
        f"configs/models/{model}.yaml",
        *CATEGORIES,
    )
display(Markdown(Path("results/metrics.md").read_text()))
display(Markdown(Path("results/robustness.md").read_text()))

## 6. Pack `results/` into one zip and download it

In [ ]:
import shutil

zip_path = shutil.make_archive(f"{WORK_DIR}/results_phase3", "zip", "results")
print("Wrote", zip_path, os.path.getsize(zip_path) // 1024, "KB")
if IN_KAGGLE:
    print("Download results_phase3.zip from the Output tab (right panel).")
else:
    from google.colab import files

    files.download(zip_path)